In [101]:
import torch
from torch import nn
from monai.networks.blocks import Convolution, MLPBlock, Upsample
from monai.networks.layers.factories import Pool

import sys
sys.path.append("/home/agustin/phd/miccai/miccai_2026/mri_x_fields/experiments/test7_my_own_net/training/scripts/networks_declaration")
# # set current directoyr as python path


from spatialattention import SpatialAttentionBlock
# import spatialattention as spatialattention

from collections.abc import Sequence

from monai.utils import ensure_tuple_rep

from monai.utils.type_conversion import convert_to_tensor

from monai.networks.layers import Act, get_pool_layer


# Generator

### Commons

In [102]:


class DiffusionUnetDownsample(nn.Module):
    """
    Downsampling layer.

    Args:
        spatial_dims: number of spatial dimensions.
        num_channels: number of input channels.
        use_conv: if True uses Convolution instead of Pool average to perform downsampling. In case that use_conv is
            False, the number of output channels must be the same as the number of input channels.
        out_channels: number of output channels.
        padding: controls the amount of implicit zero-paddings on both sides for padding number of points
            for each dimension.
    """

    def __init__(
        self, spatial_dims: int, num_channels: int, use_conv: bool, out_channels: int | None = None, padding: int = 1
    ) -> None:
        super().__init__()
        self.num_channels = num_channels
        self.out_channels = out_channels or num_channels
        self.use_conv = use_conv
        if use_conv:
            self.op = Convolution(
                spatial_dims=spatial_dims,
                in_channels=self.num_channels,
                out_channels=self.out_channels,
                strides=2,
                kernel_size=3,
                padding=padding,
                conv_only=True,
            )
        else:
            if self.num_channels != self.out_channels:
                raise ValueError("num_channels and out_channels must be equal when use_conv=False")
            self.op = Pool[Pool.AVG, spatial_dims](kernel_size=2, stride=2)

    # def forward(self, x: torch.Tensor, emb: torch.Tensor | None = None) -> torch.Tensor:
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # del emb
        if x.shape[1] != self.num_channels:
            raise ValueError(
                f"Input number of channels ({x.shape[1]}) is not equal to expected number of channels "
                f"({self.num_channels})"
            )
        output: torch.Tensor = self.op(x)
        return output


class WrappedUpsample(Upsample):
    """
    Wraps MONAI upsample block to allow for calling with timestep embeddings.
    """

    # def forward(self, x: torch.Tensor, emb: torch.Tensor | None = None) -> torch.Tensor:
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # del emb
        upsampled: torch.Tensor = super().forward(x)
        return upsampled



class DiffusionUNetResnetBlock(nn.Module):
    """
    Residual block with timestep conditioning.

    Args:
        spatial_dims: The number of spatial dimensions.
        in_channels: number of input channels.
        # temb_channels: number of timestep embedding  channels.
        out_channels: number of output channels.
        up: if True, performs upsampling.
        down: if True, performs downsampling.
        norm_num_groups: number of groups for the group normalization.
        norm_eps: epsilon for the group normalization.
    """

    def __init__(
        self,
        spatial_dims: int,
        in_channels: int,
        # temb_channels: int,
        out_channels: int | None = None,
        up: bool = False,
        down: bool = False,
        norm_num_groups: int = 32,
        norm_eps: float = 1e-6,
        cond_dim: int | None = None,
    ) -> None:
        super().__init__()
        self.spatial_dims = spatial_dims
        self.channels = in_channels
        # self.emb_channels = temb_channels
        self.out_channels = out_channels or in_channels
        self.up = up
        self.down = down

        self.norm1 = nn.GroupNorm(num_groups=norm_num_groups, num_channels=in_channels, eps=norm_eps, affine=True)
        self.nonlinearity = nn.SiLU()
        self.conv1 = Convolution(
            spatial_dims=spatial_dims,
            in_channels=in_channels,
            out_channels=self.out_channels,
            strides=1,
            kernel_size=3,
            padding=1,
            conv_only=True,
        )

        self.upsample = self.downsample = None
        if self.up:
            self.upsample = WrappedUpsample(
                spatial_dims=spatial_dims,
                mode="nontrainable",
                in_channels=in_channels,
                out_channels=in_channels,
                interp_mode="nearest",
                scale_factor=2.0,
                align_corners=None,
            )
        elif down:
            self.downsample = DiffusionUnetDownsample(spatial_dims, in_channels, use_conv=False)

        # self.time_emb_proj = nn.Linear(temb_channels, self.out_channels)
        self.cond_proj = None
        if cond_dim is not None:
            self.cond_proj = nn.Sequential(
                nn.SiLU(),
                nn.Linear(cond_dim, 2 * self.out_channels)
            )

        self.norm2 = nn.GroupNorm(num_groups=norm_num_groups, num_channels=self.out_channels, eps=norm_eps, affine=True)
        self.conv2 = Convolution(
                spatial_dims=spatial_dims,
                in_channels=self.out_channels,
                out_channels=self.out_channels,
                strides=1,
                kernel_size=3,
                padding=1,
                conv_only=True,
            )
        self.skip_connection: nn.Module
        if self.out_channels == in_channels:
            self.skip_connection = nn.Identity()
        else:
            self.skip_connection = Convolution(
                spatial_dims=spatial_dims,
                in_channels=in_channels,
                out_channels=self.out_channels,
                strides=1,
                kernel_size=1,
                padding=0,
                conv_only=True,
            )

    # def forward(self, x: torch.Tensor, emb: torch.Tensor) -> torch.Tensor:
    # def forward(self, x: torch.Tensor) -> torch.Tensor:
    def forward(self, x: torch.Tensor, cond: torch.Tensor | None = None) -> torch.Tensor:
        h = x
        h = self.norm1(h)
        h = self.nonlinearity(h)

        if self.upsample is not None:
            x = self.upsample(x)
            h = self.upsample(h)
        elif self.downsample is not None:
            x = self.downsample(x)
            h = self.downsample(h)

        h = self.conv1(h)

        # if self.spatial_dims == 2:
        #     temb = self.time_emb_proj(self.nonlinearity(emb))[:, :, None, None]
        # else:
        #     temb = self.time_emb_proj(self.nonlinearity(emb))[:, :, None, None, None]
        # h = h + temb

        h = self.norm2(h)

        if self.cond_proj is not None and cond is not None:

            gamma_beta = self.cond_proj(cond)

            gamma, beta = torch.chunk(gamma_beta, 2, dim=1)

            if self.spatial_dims == 2:
                gamma = gamma[:, :, None, None]
                beta = beta[:, :, None, None]
            else:
                gamma = gamma[:, :, None, None, None]
                beta = beta[:, :, None, None, None]

            h = h * (1 + gamma) + beta


        h = self.nonlinearity(h)
        h = self.conv2(h)
        output: torch.Tensor = self.skip_connection(x) + h
        return output



### Down Block

In [103]:

class DownBlock(nn.Module):
    """
    Unet's down block containing resnet and downsamplers blocks.

    Args:
        spatial_dims: The number of spatial dimensions.
        in_channels: number of input channels.
        out_channels: number of output channels.
        # temb_channels: number of timestep embedding channels.
        num_res_blocks: number of residual blocks.
        norm_num_groups: number of groups for the group normalization.
        norm_eps: epsilon for the group normalization.
        add_downsample: if True add downsample block.
        resblock_updown: if True use residual blocks for downsampling.
        downsample_padding: padding used in the downsampling block.
    """

    def __init__(
        self,
        spatial_dims: int,
        in_channels: int,
        out_channels: int,
        # temb_channels: int,
        num_res_blocks: int = 1,
        norm_num_groups: int = 32,
        norm_eps: float = 1e-6,
        add_downsample: bool = True,
        resblock_updown: bool = False,
        downsample_padding: int = 1,
        cond_dim: int | None = None,
    ) -> None:
        super().__init__()
        self.resblock_updown = resblock_updown

        resnets = []

        for i in range(num_res_blocks):
            in_channels = in_channels if i == 0 else out_channels
            resnets.append(
                DiffusionUNetResnetBlock(
                    spatial_dims=spatial_dims,
                    in_channels=in_channels,
                    out_channels=out_channels,
                    norm_num_groups=norm_num_groups,
                    norm_eps=norm_eps,
                    cond_dim=cond_dim,
                )
            )

        self.resnets = nn.ModuleList(resnets)

        if add_downsample:
            self.downsampler: nn.Module | None
            if resblock_updown:
                self.downsampler = DiffusionUNetResnetBlock(
                    spatial_dims=spatial_dims,
                    in_channels=out_channels,
                    out_channels=out_channels,
                    # temb_channels=temb_channels,
                    norm_num_groups=norm_num_groups,
                    norm_eps=norm_eps,
                    down=True,
                )
            else:
                self.downsampler = DiffusionUnetDownsample(
                    spatial_dims=spatial_dims,
                    num_channels=out_channels,
                    use_conv=True,
                    out_channels=out_channels,
                    padding=downsample_padding,
                )
        else:
            self.downsampler = None

    # def forward(
    #     self, hidden_states: torch.Tensor, temb: torch.Tensor, context: torch.Tensor | None = None
    # ) -> tuple[torch.Tensor, list[torch.Tensor]]:

    def forward(self, hidden_states: torch.Tensor, cond: torch.Tensor | None = None) -> tuple[torch.Tensor, list[torch.Tensor]]:
        # del context
        output_states = []

        for resnet in self.resnets:
            # hidden_states = resnet(hidden_states, temb)
            hidden_states = resnet(hidden_states, cond)
            output_states.append(hidden_states)

        if self.downsampler is not None:
            # hidden_states = self.downsampler(hidden_states, temb)
            hidden_states = self.downsampler(hidden_states)
            output_states.append(hidden_states)

        return hidden_states, output_states


### Middle block

In [104]:

class AttnMidBlock(nn.Module):
    """
    Unet's mid block containing resnet and self-attention blocks.

    Args:
        spatial_dims: The number of spatial dimensions.
        in_channels: number of input channels.
        # temb_channels: number of timestep embedding channels.
        norm_num_groups: number of groups for the group normalization.
        norm_eps: epsilon for the group normalization.
        num_head_channels: number of channels in each attention head.
        include_fc: whether to include the final linear layer. Default to True.
        use_combined_linear: whether to use a single linear layer for qkv projection, default to False.
        use_flash_attention: if True, use Pytorch's inbuilt flash attention for a memory efficient attention mechanism
            (see https://pytorch.org/docs/2.2/generated/torch.nn.functional.scaled_dot_product_attention.html).
    """

    def __init__(
        self,
        spatial_dims: int,
        in_channels: int,
        # temb_channels: int,
        norm_num_groups: int = 32,
        norm_eps: float = 1e-6,
        num_head_channels: int = 1,
        include_fc: bool = True,
        use_combined_linear: bool = False,
        use_flash_attention: bool = False,
        cond_dim: int | None = None,
    ) -> None:
        super().__init__()

        self.resnet_1 = DiffusionUNetResnetBlock(
            spatial_dims=spatial_dims,
            in_channels=in_channels,
            out_channels=in_channels,
            # temb_channels=temb_channels,
            norm_num_groups=norm_num_groups,
            norm_eps=norm_eps,
            cond_dim=cond_dim
        )
        self.attention = SpatialAttentionBlock(
            spatial_dims=spatial_dims,
            num_channels=in_channels,
            num_head_channels=num_head_channels,
            norm_num_groups=norm_num_groups,
            norm_eps=norm_eps,
            include_fc=include_fc,
            use_combined_linear=use_combined_linear,
            use_flash_attention=use_flash_attention,
        )

        self.resnet_2 = DiffusionUNetResnetBlock(
            spatial_dims=spatial_dims,
            in_channels=in_channels,
            out_channels=in_channels,
            # temb_channels=temb_channels,
            norm_num_groups=norm_num_groups,
            norm_eps=norm_eps,
            cond_dim=cond_dim
        )

    # def forward(
    #     self, hidden_states: torch.Tensor, temb: torch.Tensor, context: torch.Tensor | None = None
    # ) -> torch.Tensor:

    def forward(self, hidden_states: torch.Tensor, cond: torch.Tensor | None = None) -> torch.Tensor:
        # del context
        # hidden_states = self.resnet_1(hidden_states, temb)
        hidden_states = self.resnet_1(hidden_states, cond)
        hidden_states = self.attention(hidden_states).contiguous()
        # hidden_states = self.resnet_2(hidden_states, temb)
        hidden_states = self.resnet_2(hidden_states, cond)

        return hidden_states


### Up Block

In [105]:

class UpBlock(nn.Module):
    """
    Unet's up block containing resnet and upsamplers blocks.

    Args:
        spatial_dims: The number of spatial dimensions.
        in_channels: number of input channels.
        prev_output_channel: number of channels from residual connection.
        out_channels: number of output channels.
        # temb_channels: number of timestep embedding channels.
        num_res_blocks: number of residual blocks.
        norm_num_groups: number of groups for the group normalization.
        norm_eps: epsilon for the group normalization.
        add_upsample: if True add downsample block.
        resblock_updown: if True use residual blocks for upsampling.
    """

    def __init__(
        self,
        spatial_dims: int,
        in_channels: int,
        prev_output_channel: int,
        out_channels: int,
        # temb_channels: int,
        num_res_blocks: int = 1,
        norm_num_groups: int = 32,
        norm_eps: float = 1e-6,
        add_upsample: bool = True,
        resblock_updown: bool = False,
        cond_dim: int | None = None,
    ) -> None:
        super().__init__()
        self.resblock_updown = resblock_updown
        resnets = []

        for i in range(num_res_blocks):
            res_skip_channels = in_channels if (i == num_res_blocks - 1) else out_channels
            resnet_in_channels = prev_output_channel if i == 0 else out_channels

            resnets.append(
                DiffusionUNetResnetBlock(
                    spatial_dims=spatial_dims,
                    in_channels=resnet_in_channels + res_skip_channels,
                    out_channels=out_channels,
                    # temb_channels=temb_channels,
                    norm_num_groups=norm_num_groups,
                    norm_eps=norm_eps,
                    cond_dim=cond_dim
                )
            )

        self.resnets = nn.ModuleList(resnets)

        self.upsampler: nn.Module | None
        if add_upsample:
            if resblock_updown:
                self.upsampler = DiffusionUNetResnetBlock(
                    spatial_dims=spatial_dims,
                    in_channels=out_channels,
                    out_channels=out_channels,
                    # temb_channels=temb_channels,
                    norm_num_groups=norm_num_groups,
                    norm_eps=norm_eps,
                    up=True,
                )
            else:
                post_conv = Convolution(
                    spatial_dims=spatial_dims,
                    in_channels=out_channels,
                    out_channels=out_channels,
                    strides=1,
                    kernel_size=3,
                    padding=1,
                    conv_only=True,
                )
                self.upsampler = WrappedUpsample(
                    spatial_dims=spatial_dims,
                    mode="nontrainable",
                    in_channels=out_channels,
                    out_channels=out_channels,
                    interp_mode="nearest",
                    scale_factor=2.0,
                    post_conv=post_conv,
                    align_corners=None,
                )

        else:
            self.upsampler = None

    # def forward(
    #     self,
    #     hidden_states: torch.Tensor,
    #     res_hidden_states_list: list[torch.Tensor],
    #     temb: torch.Tensor,
    #     context: torch.Tensor | None = None,
    # ) -> torch.Tensor:
        
    def forward(
        self,
        hidden_states: torch.Tensor,
        res_hidden_states_list: list[torch.Tensor],
        cond: torch.Tensor | None = None,
    ) -> torch.Tensor:
        # del context
        for resnet in self.resnets:
            # pop res hidden states
            res_hidden_states = res_hidden_states_list[-1]
            res_hidden_states_list = res_hidden_states_list[:-1]
            hidden_states = torch.cat([hidden_states, res_hidden_states], dim=1)

            # hidden_states = resnet(hidden_states, temb)
            hidden_states = resnet(hidden_states, cond)

        if self.upsampler is not None:
            # hidden_states = self.upsampler(hidden_states, temb)
            hidden_states = self.upsampler(hidden_states)

        return hidden_states



### Full network

In [ ]:


class MultiResolutionGenerator(nn.Module):
    """
    U-Net network with timestep embedding and attention mechanisms for conditioning based on
    Rombach et al. "High-Resolution Image Synthesis with Latent Diffusion Models" https://arxiv.org/abs/2112.10752
    and Pinaya et al. "Brain Imaging Generation with Latent Diffusion Models" https://arxiv.org/abs/2209.07162

    Args:
        spatial_dims: Number of spatial dimensions.
        in_channels: Number of input channels.
        out_channels: Number of output channels.
        num_res_blocks: Number of residual blocks (see ResnetBlock) per level. Can be a single integer or a sequence of integers.
        num_channels: Tuple of block output channels.
        attention_levels: List of levels to add attention.
        norm_num_groups: Number of groups for the normalization.
        norm_eps: Epsilon for the normalization.
        resblock_updown: If True, use residual blocks for up/downsampling.
        upcast_attention: If True, upcast attention operations to full precision.
        include_fc: whether to include the final linear layer. Default to False.
        use_combined_linear: whether to use a single linear layer for qkv projection, default to False.
        use_flash_attention: If True, use flash attention for a memory efficient attention mechanism.
    """

    def __init__(
        self,
        spatial_dims: int,
        in_channels: int,
        out_channels: int,
        num_res_blocks: Sequence[int] | int = (2, 2, 2, 2),
        num_channels: Sequence[int] = (32, 64, 64, 64),
        norm_num_groups: int = 32,
        norm_eps: float = 1e-6,
        resblock_updown: bool = False,
        include_fc: bool = False,
        use_combined_linear: bool = False,
        use_flash_attention: bool = False,
        nb_resolutions: int = 5,
        resolution_emb_channels: int | None = None,
    ) -> None:
        # print("instantiating DiffusionModelUNetMaisi")
        super().__init__()


        # All number of channels should be multiple of num_groups
        if any((out_channel % norm_num_groups) != 0 for out_channel in num_channels):
            raise ValueError(
                f"DiffusionModelUNetMaisi expects all num_channels being multiple of norm_num_groups, "
                f"but get num_channels: {num_channels} and norm_num_groups: {norm_num_groups}"
            )

        if isinstance(num_res_blocks, int):
            num_res_blocks = ensure_tuple_rep(num_res_blocks, len(num_channels))

        if len(num_res_blocks) != len(num_channels):
            raise ValueError(
                "`num_res_blocks` should be a single integer or a tuple of integers with the same length as "
                "`num_channels`."
            )

        if use_flash_attention is True and not torch.cuda.is_available():
            raise ValueError(
                "torch.cuda.is_available() should be True but is False. Flash attention is only available for GPU."
            )

        self.in_channels = in_channels
        self.block_out_channels = num_channels
        self.out_channels = out_channels
        self.num_res_blocks = num_res_blocks

        # conditioning dimension
        self.field_embedding = None
        if resolution_emb_channels is not None:
            self.field_embedding = nn.Embedding(nb_resolutions, resolution_emb_channels)

        # input
        self.conv_in = Convolution(
            spatial_dims=spatial_dims,
            in_channels=in_channels,
            out_channels=num_channels[0],
            strides=1,
            kernel_size=3,
            padding=1,
            conv_only=True,
        )

        # down
        self.down_blocks = nn.ModuleList([])
        output_channel = num_channels[0]
        for i in range(len(num_channels)):
            input_channel = output_channel
            output_channel = num_channels[i]
            is_final_block = i == len(num_channels) - 1
            down_block = DownBlock(
                spatial_dims=spatial_dims,
                in_channels=input_channel,
                out_channels=output_channel,
                num_res_blocks=num_res_blocks[i],
                norm_num_groups=norm_num_groups,
                norm_eps=norm_eps,
                add_downsample=not is_final_block,
                resblock_updown=resblock_updown,
                cond_dim=resolution_emb_channels,
            )

            self.down_blocks.append(down_block)

        # mid
        self.middle_block = AttnMidBlock(
            spatial_dims=spatial_dims,
            in_channels=num_channels[-1],
            norm_num_groups=norm_num_groups,
            norm_eps=norm_eps,
            num_head_channels=1,
            include_fc=include_fc,
            use_combined_linear=use_combined_linear,
            use_flash_attention=use_flash_attention,
            cond_dim=resolution_emb_channels
        )

        # up
        self.up_blocks = nn.ModuleList([])
        reversed_block_out_channels = list(reversed(num_channels))
        reversed_num_res_blocks = list(reversed(num_res_blocks))
        output_channel = reversed_block_out_channels[0]
        for i in range(len(reversed_block_out_channels)):
            prev_output_channel = output_channel
            output_channel = reversed_block_out_channels[i]
            input_channel = reversed_block_out_channels[min(i + 1, len(num_channels) - 1)]

            is_final_block = i == len(num_channels) - 1

            up_block = UpBlock(
                spatial_dims=spatial_dims,
                in_channels=input_channel,
                prev_output_channel=prev_output_channel,
                out_channels=output_channel,
                num_res_blocks=reversed_num_res_blocks[i] + 1,
                norm_num_groups=norm_num_groups,
                norm_eps=norm_eps,
                add_upsample=not is_final_block,
                resblock_updown=resblock_updown,
                # cond_dim=resolution_emb_channels # No conditioning for up blocks, as we only condition on the down blocks and mid block
            )

            self.up_blocks.append(up_block)



        # resolution_residuals
        self.resolution_residuals_list = nn.ModuleList([])
        res_residual_in_channels = num_channels[0]

        for i in range(nb_resolutions-1):
            res_residual_out_channels = res_residual_in_channels * 2

            out =  DiffusionUNetResnetBlock(
                    spatial_dims=spatial_dims,
                    in_channels=res_residual_in_channels,
                    out_channels=res_residual_out_channels,
                    # temb_channels=temb_channels,
                    norm_num_groups=norm_num_groups,
                    norm_eps=norm_eps,
                )
            self.resolution_residuals_list.append(out)

            res_residual_in_channels = res_residual_out_channels

        # out # 
        self.out_list = nn.ModuleList([])
        out_in_channels = num_channels[0]
        for i in range(nb_resolutions):
            out = nn.Sequential(
                nn.GroupNorm(num_groups=norm_num_groups, num_channels=out_in_channels, eps=norm_eps, affine=True),
                nn.SiLU(),
                Convolution(
                    spatial_dims=spatial_dims,
                    in_channels=out_in_channels,
                    out_channels=out_channels,
                    strides=1,
                    kernel_size=3,
                    padding=1,
                    conv_only=True,
                )
            )
            self.out_list.append(out)
            out_in_channels = out_in_channels * 2

    def _apply_down_blocks(self, h, cond=None):
        down_block_res_samples: list[torch.Tensor] = [h]
        for i, downsample_block in enumerate(self.down_blocks):
            h, res_samples = downsample_block(hidden_states=h, cond=cond)
            down_block_res_samples.extend(res_samples)


        return h, down_block_res_samples




    def _apply_up_blocks(self, h, down_block_res_samples):
        for upsample_block in self.up_blocks:
            res_samples = down_block_res_samples[-len(upsample_block.resnets) :]
            down_block_res_samples = down_block_res_samples[: -len(upsample_block.resnets)]
            h = upsample_block(hidden_states=h, res_hidden_states_list=res_samples)

        return h
    

    def _apply_resolution_residuals(self, h):
        resolution_residuals: list[torch.Tensor] = [h]
        for res_block in self.resolution_residuals_list:
            h = res_block(h)
            resolution_residuals.append(h)
        return resolution_residuals

    def _apply_out_blocks(self, resolution_residuals):
        out_samples: list[torch.Tensor] = []
        for i, out_block in enumerate(self.out_list):
            h = resolution_residuals[i]
            h = out_block(h)
            out_samples.append(convert_to_tensor(h))
        return out_samples
    
    def forward(
        self,
        x: torch.Tensor,
        cond: torch.Tensor | None = None,
    ) -> torch.Tensor:
        """
        Forward pass through the UNet model.

        Args:
            x: Input tensor of shape (N, C, SpatialDims).
        Returns:
            A tensor representing the output of the UNet model.
        """

        # condition embedding
        cond_emb = None
        if self.field_embedding is not None and cond is not None:
            cond_emb = self.field_embedding(cond)

        h = self.conv_in(x)
        h, _updated_down_block_res_samples = self._apply_down_blocks(h, cond_emb)
        h = self.middle_block(h, cond_emb)
        h = self._apply_up_blocks(h, _updated_down_block_res_samples)

        resolution_residuals = self._apply_resolution_residuals(h)
        outs = self._apply_out_blocks(resolution_residuals)
        return outs



# Discriminator

In [109]:

class PatchDiscriminator(nn.Sequential):
    """
    PatchGAN + ACGAN discriminator for MRI domain conditioning.

    Args:
        spatial_dims: number of spatial dimensions (1D, 2D etc.)
        num_channels: number of filters in the first convolutional layer (double of the value is taken from then on)
        in_channels: number of input channels
        out_channels: number of output channels in each discriminator
        num_layers_d: number of Convolution layers (Conv + activation + normalisation + [dropout]) in each
            of the discriminators. In each layer, the number of channels are doubled and the spatial size is
            divided by 2.
        kernel_size: kernel size of the convolution layers
        activation: activation layer type
        norm: normalisation type
        bias: introduction of layer bias
        padding: padding to be applied to the convolutional layers
        dropout: proportion of dropout applied, defaults to 0.
        last_conv_kernel_size: kernel size of the last convolutional layer.

        Outputs:
        - patch_logits: (B, 1, H', W', D') realism map
        - class_logits: (B, num_classes) scanner classification
        - features: intermediate feature maps (for feature matching)
    """

    def __init__(
        self,
        spatial_dims: int,
        num_channels: int,
        in_channels: int,
        out_channels: int = 1,
        num_layers_d: int = 3,
        num_classes: int = 5,  # 0.1T, 1.5T, 3T, 5T, 7T
        kernel_size: int = 4,
        activation: str | tuple = (Act.LEAKYRELU, {"negative_slope": 0.2}),
        norm: str | tuple = "GROUP",
        bias: bool = False,
        padding: int | Sequence[int] = 1,
        dropout: float | tuple = 0.0,
        last_conv_kernel_size: int | None = None,
    ) -> None: 
        super().__init__()
        if last_conv_kernel_size is None:
            last_conv_kernel_size = kernel_size
        self.num_layers_d = num_layers_d

        # -------------------------
        # Shared convolutional trunk
        # -------------------------
        layers = []

        layers.append(
            Convolution(
                spatial_dims=spatial_dims,
                kernel_size=kernel_size,
                in_channels=in_channels,
                out_channels=num_channels,
                act=activation,
                bias=True,
                norm=None,
                dropout=dropout,
                padding=padding,
                strides=2,
            )
        )

        input_ch = num_channels
        output_ch = num_channels * 2

        for i in range(num_layers_d):
            stride = 1 if i == num_layers_d - 1 else 2

            layers.append(
                Convolution(
                    spatial_dims=spatial_dims,
                    kernel_size=kernel_size,
                    in_channels=input_ch,
                    out_channels=output_ch,
                    act=activation,
                    bias=bias,
                    norm=norm,
                    dropout=dropout,
                    padding=padding,
                    strides=stride,
                )
            )

            input_ch = output_ch
            output_ch *= 2

        self.backbone = nn.Sequential(*layers)

        # -------------------------
        # PatchGAN head (real/fake per patch)
        # -------------------------
        self.patch_head = Convolution(
            spatial_dims=spatial_dims,
            kernel_size=last_conv_kernel_size,
            in_channels=input_ch,
            out_channels=out_channels,
            conv_only=True,
            padding=1,
            strides=1,
        )

        # -------------------------
        # ACGAN head (scanner classification)
        # -------------------------
        self.global_pool = nn.AdaptiveAvgPool3d(1) if spatial_dims == 3 else nn.AdaptiveAvgPool2d(1)

        self.classifier = nn.Sequential(
            nn.Linear(input_ch, 128),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(128, num_classes),
        )

        self.apply(self._init_weights)

    # -------------------------
    # forward
    # -------------------------
    def forward(self, x):
        feats = self.backbone(x)

        # Patch realism logits
        patch_logits = self.patch_head(feats)

        # Global classification logits (scanner type)
        pooled = self.global_pool(feats)
        pooled = pooled.view(pooled.shape[0], -1)
        class_logits = self.classifier(pooled)

        return {
            "patch_logits": patch_logits,
            "class_logits": class_logits,
            "features": feats
        }

    # -------------------------
    # initialization
    # -------------------------
    def _init_weights(self, m):
        if isinstance(m, nn.Conv2d) or isinstance(m, nn.Conv3d):
            nn.init.normal_(m.weight, 0.0, 0.02)
        elif isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, 0.0, 0.02)
            nn.init.constant_(m.bias, 0)

In [112]:
device_name = f"cuda:3"
device = torch.device(device_name)

spatial_dims = 3
in_channels = 4
out_channels = 4
num_res_blocks = 2
num_channels = [32,64,128,256] #[8,16,32,64]
norm_num_groups = 8
norm_eps = 1e-6
resblock_updown = False
include_fc = False
use_flash_attention = True

generator = MultiResolutionGenerator(
    spatial_dims=spatial_dims,
    in_channels=in_channels,
    out_channels=out_channels,
    num_res_blocks=num_res_blocks,
    num_channels=num_channels,
    norm_num_groups=norm_num_groups,
    norm_eps=norm_eps,
    resblock_updown=resblock_updown,
    include_fc=include_fc,
    use_flash_attention=use_flash_attention,
    nb_resolutions=5,
    resolution_emb_channels=64
)

discriminator = PatchDiscriminator(
    spatial_dims=spatial_dims,
    num_channels=32,
    in_channels=out_channels,
    out_channels=out_channels,
    num_layers_d=3,
    kernel_size=4,
    activation=(Act.LEAKYRELU, {"negative_slope": 0.2}),
    norm="INSTANCE",
    bias=False,
    padding=1,
    dropout=0.0,
    last_conv_kernel_size=4,
)


print("Gen number of parameters: ", sum(p.numel() for p in generator.parameters() if p.requires_grad))
print("Disc number of parameters: ", sum(p.numel() for p in discriminator.parameters() if p.requires_grad))

# input = torch.randn(1, in_channels, 64,64,64)  # Example input tensor with shape (N, C, D, H, W)
input = torch.randn(1, in_channels, 96,112,96)  # Example input tensor with shape (N, C, D, H, W)
input = input.to(device)
generator = generator.to(device)
discriminator = discriminator.to(device)
output_g = generator(input)  # Forward pass through the generator model
output_d = discriminator(input)

print("Gen Output shape: ", [o.shape for o in output_g])  # Print the shape of the output tensor
print("Disc Output shape: ", [o.shape for o in output_d["patch_logits"]])  # Print the shape of the output tensor
print("Disc Output shape: ", [o.shape for o in output_d["class_logits"]])  # Print the shape of the output tensor
print("Disc Output shape: ", [o.shape for o in output_d["features"]])  # Print the shape of the output tensor
# print("Disc Output shape: ", [o.shape for o in output_d])  # Print the shape of the output tensor
# print(generator)
# print(discriminator)

Gen number of parameters:  54204660
Disc number of parameters:  2859817
Gen Output shape:  [torch.Size([1, 4, 96, 112, 96]), torch.Size([1, 4, 96, 112, 96]), torch.Size([1, 4, 96, 112, 96]), torch.Size([1, 4, 96, 112, 96]), torch.Size([1, 4, 96, 112, 96])]
Disc Output shape:  [torch.Size([4, 10, 12, 10])]
Disc Output shape:  [torch.Size([5])]
Disc Output shape:  [torch.Size([256, 11, 13, 11])]


# Training

### Training Config

In [ ]:
# {
#     "data_option":{
#         "random_aug": true,
#         "spacing_type": "rand_zoom",
#         "spacing": null,
#         "select_channel": 0
#     },
#     "autoencoder_train": {
#         "batch_size": 1,
#         "patch_size": [64, 64, 64],
#         "val_batch_size": 1,
#         "val_patch_size": null,
#         "val_sliding_window_patch_size": [96, 96, 64],
#         "lr": 1e-4,
#         "perceptual_weight": 0.3,
#         "kl_weight": 1e-7,
#         "adv_weight": 0.1,
#         "recon_loss": "l1",
#         "val_interval": 10,
#         "cache": 0.5,
#         "amp": true,
#         "n_epochs": 1
#     }
# }


In [ ]:
from torch.nn import L1Loss, MSELoss
from monai.losses.adversarial_loss import PatchAdversarialLoss
from torch.optim import lr_scheduler
from torch.amp import GradScaler, autocast


amp = True
lr = 1e-4
recon_loss = "l2"
eps = 1e-6 if amp else 1e-8
# config loss and loss weight
if recon_loss == "l2":
    intensity_loss = MSELoss()
    print("Use l2 loss")
else:
    intensity_loss = L1Loss(reduction="mean")
    print("Use l1 loss")
adv_loss = PatchAdversarialLoss(criterion="least_squares")


# Not used becuase latent space can cause problems, try it later
# loss_perceptual = (
#     PerceptualLoss(spatial_dims=3, network_type="squeeze", is_fake_3d=True, fake_3d_ratio=0.2).eval().to(device)
# )

# config optimizer and lr scheduler
optimizer_g = torch.optim.Adam(params=generator.parameters(), lr=lr, eps=1e-06 if amp else 1e-08)
optimizer_d = torch.optim.Adam(params=discriminator.parameters(), lr=lr, eps=1e-06 if amp else 1e-08)


# please adjust the learning rate warmup rule based on your dataset and n_epochs
def warmup_rule(step):
    # learning rate warmup rule
    if step < 1000:
        return 0.01
    elif step < 2000:
        return 0.1
    else:
        return 1.0


scheduler_g = lr_scheduler.LambdaLR(optimizer_g, lr_lambda=warmup_rule)
scheduler_d = lr_scheduler.LambdaLR(optimizer_d, lr_lambda=warmup_rule)

# set AMP scaler
if amp:
    # test use mean reduction for everything
    scaler_g = GradScaler("cuda", init_scale=2.0**8, growth_factor=1.5)
    scaler_d = GradScaler("cuda", init_scale=2.0**8, growth_factor=1.5) 

### Training Loop

In [ ]:
# Initialize variables
val_interval = args.val_interval
best_val_recon_epoch_loss = 10000000.0
total_step = 0
start_epoch = 0
max_epochs = args.n_epochs

# Setup validation inferer
# val_inferer = (
#     SlidingWindowInferer(
#         roi_size=args.val_sliding_window_patch_size,
#         sw_batch_size=1,
#         progress=False,
#         overlap=0.0,
#         device=torch.device("cpu"),
#         sw_device=device,
#     )
#     if args.val_sliding_window_patch_size
#     else SimpleInferer()
# )


def loss_weighted_sum(losses):
    return losses["recons_loss"] + args.kl_weight * losses["kl_loss"] + args.perceptual_weight * losses["p_loss"]


# Training and validation loops
for epoch in range(start_epoch, max_epochs):
    print("lr:", scheduler_g.get_lr())
    autoencoder.train()
    discriminator.train()
    train_epoch_losses = {"recons_loss": 0, "kl_loss": 0, "p_loss": 0}

    for batch in dataloader_train:
        images = batch["image"].to(device).contiguous()
        optimizer_g.zero_grad(set_to_none=True)
        optimizer_d.zero_grad(set_to_none=True)
        with autocast("cuda", enabled=args.amp):
            # Train Generator
            reconstruction, z_mu, z_sigma = autoencoder(images)
            losses = {
                "recons_loss": intensity_loss(reconstruction, images),
                "kl_loss": KL_loss(z_mu, z_sigma),
                "p_loss": loss_perceptual(reconstruction.float(), images.float()),
            }
            logits_fake = discriminator(reconstruction.contiguous().float())[-1]
            generator_loss = adv_loss(logits_fake, target_is_real=True, for_discriminator=False)
            loss_g = loss_weighted_sum(losses) + args.adv_weight * generator_loss

            if args.amp:
                scaler_g.scale(loss_g).backward()
                scaler_g.unscale_(optimizer_g)
                scaler_g.step(optimizer_g)
                scaler_g.update()
            else:
                loss_g.backward()
                optimizer_g.step()

            # Train Discriminator
            logits_fake = discriminator(reconstruction.contiguous().detach())[-1]
            loss_d_fake = adv_loss(logits_fake, target_is_real=False, for_discriminator=True)
            logits_real = discriminator(images.contiguous().detach())[-1]
            loss_d_real = adv_loss(logits_real, target_is_real=True, for_discriminator=True)
            loss_d = (loss_d_fake + loss_d_real) * 0.5

            if args.amp:
                scaler_d.scale(loss_d).backward()
                scaler_d.step(optimizer_d)
                scaler_d.update()
            else:
                loss_d.backward()
                optimizer_d.step()

        # Log training loss
        total_step += 1
        for loss_name, loss_value in losses.items():
            tensorboard_writer.add_scalar(f"train_{loss_name}_iter", loss_value.item(), total_step)
            train_epoch_losses[loss_name] += loss_value.item()
        tensorboard_writer.add_scalar("train_adv_loss_iter", generator_loss, total_step)
        tensorboard_writer.add_scalar("train_fake_loss_iter", loss_d_fake, total_step)
        tensorboard_writer.add_scalar("train_real_loss_iter", loss_d_real, total_step)

    scheduler_g.step()
    scheduler_d.step()
    for key in train_epoch_losses:
        train_epoch_losses[key] /= len(dataloader_train)
    print(f"Epoch {epoch} train_vae_loss {loss_weighted_sum(train_epoch_losses)}: {train_epoch_losses}.")
    for loss_name, loss_value in train_epoch_losses.items():
        tensorboard_writer.add_scalar(f"train_{loss_name}_epoch", loss_value, epoch)
    torch.save(autoencoder.state_dict(), trained_g_path)
    torch.save(discriminator.state_dict(), trained_d_path)
    print("Save trained autoencoder to", trained_g_path)
    print("Save trained discriminator to", trained_d_path)

    # Validation
    if epoch % val_interval == 0:
        autoencoder.eval()
        val_epoch_losses = {"recons_loss": 0, "kl_loss": 0, "p_loss": 0}
        val_loader_iter = iter(dataloader_val)
        for batch in dataloader_val:
            with torch.no_grad():
                with autocast("cuda", enabled=args.amp):
                    images = batch["image"]
                    reconstruction, _, _ = dynamic_infer(val_inferer, autoencoder, images)
                    reconstruction = reconstruction.to(device)
                    val_epoch_losses["recons_loss"] += intensity_loss(reconstruction, images.to(device)).item()
                    val_epoch_losses["kl_loss"] += KL_loss(z_mu, z_sigma).item()
                    val_epoch_losses["p_loss"] += loss_perceptual(reconstruction, images.to(device)).item()

        for key in val_epoch_losses:
            val_epoch_losses[key] /= len(dataloader_val)

        val_loss_g = loss_weighted_sum(val_epoch_losses)
        print(f"Epoch {epoch} val_vae_loss {val_loss_g}: {val_epoch_losses}.")

        if val_loss_g < best_val_recon_epoch_loss:
            best_val_recon_epoch_loss = val_loss_g
            trained_g_path_epoch = f"{trained_g_path[:-3]}_epoch{epoch}.pt"
            torch.save(autoencoder.state_dict(), trained_g_path_epoch)
            print("Got best val vae loss.")
            print("Save trained autoencoder to", trained_g_path_epoch)

        for loss_name, loss_value in val_epoch_losses.items():
            tensorboard_writer.add_scalar(loss_name, loss_value, epoch)

        # Monitor scale_factor
        # We'd like to tune kl_weights in order to make scale_factor close to 1.
        scale_factor_sample = 1.0 / z_mu.flatten().std()
        tensorboard_writer.add_scalar("val_one_sample_scale_factor", scale_factor_sample, epoch)

        # Monitor reconstruction result
        center_loc_axis = find_label_center_loc(images[0, 0, ...])
        vis_image = get_xyz_plot(images[0, ...], center_loc_axis, mask_bool=False)
        vis_recon_image = get_xyz_plot(reconstruction[0, ...], center_loc_axis, mask_bool=False)

        tensorboard_writer.add_image(
            "val_orig_img",
            vis_image.transpose([2, 0, 1]),
            epoch,
        )
        tensorboard_writer.add_image(
            "val_recon_img",
            vis_recon_image.transpose([2, 0, 1]),
            epoch,
        )

        show_image(vis_image, title="val image")
        show_image(vis_recon_image, title="val recon result")

In [58]:


input = torch.randn(1, out_channels, 96,112,96)  # Example input tensor with shape (N, C, D, H, W)
output = discriminator(input)  # Forward pass through the discriminator model
print("Output shape: ", [o.shape for o in output])  # Print the shape of the output tensor
# print("number of parameters: ", sum(p.numel() for p in discriminator.parameters() if p.requires_grad))
# print(discriminator)

Output shape:  [torch.Size([1, 32, 48, 56, 48]), torch.Size([1, 64, 24, 28, 24]), torch.Size([1, 128, 23, 27, 23]), torch.Size([1, 4, 22, 26, 22])]


In [34]:
spatial_dims = 3
in_channels = 4
out_channels = 8
num_res_blocks = 1
norm_num_groups = 4
norm_eps = 1e-6
add_downsample = True
resblock_updown = False

# class Encoder(nn.Module):
#     def __init__(self):
#         super().__init__(

#         )
#         self.encoder = DownBlock(
#             spatial_dims=spatial_dims,
#             in_channels=in_channels,
#             out_channels=out_channels,
#             # temb_channels=temb_channels,
#             num_res_blocks=num_res_blocks,
#             norm_num_groups=norm_num_groups,
#             norm_eps=norm_eps,
#             add_downsample=add_downsample,
#             resblock_updown=resblock_updown,
#         )

#     def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, list[torch.Tensor]]:
#         return self.encoder(x)
    

# encoder = DownBlock(
#         spatial_dims=spatial_dims,
#         in_channels=in_channels,
#         out_channels=out_channels,
#         # temb_channels=temb_channels,
#         num_res_blocks=num_res_blocks,
#         norm_num_groups=norm_num_groups,
#         norm_eps=norm_eps,
#         add_downsample=add_downsample,
#         resblock_updown=resblock_updown,
#     )


in_channels = 64
norm_num_groups = 32
mid_block = AttnMidBlock(
    spatial_dims=spatial_dims,
    in_channels=in_channels,
    # temb_channels=temb_channels,
    norm_num_groups=norm_num_groups,
    norm_eps=norm_eps,
    num_head_channels=1,
    include_fc=False,
    use_combined_linear=False,
    use_flash_attention=True,
)

print(mid_block)

AttnMidBlock(
  (resnet_1): DiffusionUNetResnetBlock(
    (norm1): GroupNorm(32, 64, eps=1e-06, affine=True)
    (nonlinearity): SiLU()
    (conv1): Convolution(
      (conv): Conv3d(64, 64, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
    )
    (norm2): GroupNorm(32, 64, eps=1e-06, affine=True)
    (conv2): Convolution(
      (conv): Conv3d(64, 64, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
    )
    (skip_connection): Identity()
  )
  (attention): SpatialAttentionBlock(
    (norm): GroupNorm(32, 64, eps=1e-06, affine=True)
    (attn): SABlock(
      (out_proj): Linear(in_features=64, out_features=64, bias=True)
      (to_q): Linear(in_features=64, out_features=64, bias=True)
      (to_k): Linear(in_features=64, out_features=64, bias=True)
      (to_v): Linear(in_features=64, out_features=64, bias=True)
      (qkv): Identity()
      (input_rearrange): Rearrange('b h (l d) -> b l h d', l=64)
      (out_rearrange): Rearrange('b l h d -> b h (l d)')
      